# LatAm LC Spread PCA Decomposition
PCA on local currency sovereign spreads vs USTs — country dislocation diagnostics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## Cell 1 — Parameters

In [ ]:
# --- tenor node to analyze ---
TENOR = '10Y'

# --- country map: display_name -> DataFrame ---
COUNTRIES = {
    'Peru':     df_perugb_cmt,
    'Mexico':   df_mbono_cmt,
    'Colombia': df_coltes_cmt,
    'Chile':    df_btpcl_cmt,
    'Brazil':   df_bntnf_cmt,
}

# --- country for detailed decomposition charts ---
FOCUS_COUNTRY = 'Peru'

# --- PCA estimation window start (None = full history) ---
LOOKBACK_START = '2022-01-01'

# --- number of PCs to retain (None = auto via 90% variance threshold) ---
N_PCS = 3

# --- rolling window (bus days) for residual SD bands ---
SD_WINDOW = 252

# --- SD thresholds for band shading ---
SD_BANDS = [1.25, 1.65]

# --- rolling sum windows for factor returns ---
ROLLING_WINDOWS = [5, 10, 20]

# --- which windows to actually plot (subset of ROLLING_WINDOWS) ---
ROLL_DISPLAY = [5, 20]

## Cell 2 — Build Spread Panel

In [ ]:
def build_spread_panel(countries, tenor, ust_df):
    # align UST series on Fecha index
    ust = ust_df.set_index('Fecha')[tenor].rename('UST')
    frames = {}
    for name, df in countries.items():
        loc = df.set_index('Fecha')[tenor]
        # intersect dates, compute spread in bps
        idx = loc.index.intersection(ust.index)
        frames[name] = (loc.loc[idx] - ust.loc[idx]) * 100
    panel = pd.DataFrame(frames)
    panel.index.name = 'Fecha'
    return panel.sort_index()

df_spreads = build_spread_panel(COUNTRIES, TENOR, df_ust_cmt)
print(f'Spread panel: {df_spreads.shape[0]} dates x {df_spreads.shape[1]} countries')
print(f'Date range: {df_spreads.index[0].date()} to {df_spreads.index[-1].date()}')
df_spreads.tail(3)

## Cell 3 — PCA Engine

In [ ]:
def run_pca(df_spreads, lookback_start, n_pcs):
    # filter to estimation window
    data = df_spreads.copy()
    if lookback_start is not None:
        data = data.loc[lookback_start:]
    data = data.dropna()
    # demean
    means = data.mean()
    dm = data - means
    # covariance and eigen decomposition
    cov = dm.cov().values
    eigvals, eigvecs = np.linalg.eigh(cov)
    # sort descending
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    # variance explained
    total_var = eigvals.sum()
    var_exp = eigvals / total_var
    cum_var = np.cumsum(var_exp)
    # select n_pcs
    if n_pcs is None:
        n_pcs = int(np.searchsorted(cum_var, 0.90)) + 1
    n_pcs = min(n_pcs, len(eigvals))
    # loadings and scores
    loadings = pd.DataFrame(
        eigvecs[:, :n_pcs],
        index=data.columns,
        columns=[f'PC{i+1}' for i in range(n_pcs)]
    )
    scores = pd.DataFrame(
        dm.values @ eigvecs[:, :n_pcs],
        index=dm.index,
        columns=[f'PC{i+1}' for i in range(n_pcs)]
    )
    return {
        'means':         means,
        'loadings':      loadings,
        'scores':        scores,
        'eigenvalues':   eigvals,
        'var_explained': var_exp,
        'cum_var_explained': cum_var,
        'n_pcs':         n_pcs,
        'demeaned':      dm,
    }

pca = run_pca(df_spreads, LOOKBACK_START, N_PCS)
print(f'n_pcs retained: {pca["n_pcs"]}')
print(f'Var explained: {[f"{v:.1%}" for v in pca["var_explained"][:pca["n_pcs"]]]}')
print(f'Cum var: {pca["cum_var_explained"][pca["n_pcs"]-1]:.1%}')

## Cell 4 — Fitted Spreads, Residuals, Per-Factor Contributions

In [ ]:
def decompose(pca, df_spreads):
    means    = pca['means']
    loadings = pca['loadings']   # countries x n_pcs
    n_pcs    = pca['n_pcs']
    pcs      = loadings.columns.tolist()
    # project all dates onto stored loadings
    dm_full  = df_spreads.subtract(means).dropna()
    scores_full = pd.DataFrame(
        dm_full.values @ loadings.values,
        index=dm_full.index,
        columns=pcs
    )
    # reconstruct fitted spreads
    fitted = pd.DataFrame(
        scores_full.values @ loadings.values.T + means.values,
        index=dm_full.index,
        columns=means.index
    )
    # residuals
    actual    = df_spreads.loc[dm_full.index]
    residuals = actual - fitted
    # per-factor contributions: score_k * loading_country_k  for each country
    contributions = {}
    for country in means.index:
        contrib = pd.DataFrame(index=scores_full.index, columns=pcs, dtype=float)
        for pc in pcs:
            contrib[pc] = scores_full[pc] * loadings.loc[country, pc]
        contributions[country] = contrib
    return {
        'fitted':       fitted,
        'residuals':    residuals,
        'contributions': contributions,
        'scores_full':  scores_full,
    }

decomp = decompose(pca, df_spreads)
print('Decomposition complete.')
print(f'Fitted range:   {decomp["fitted"].index[0].date()} to {decomp["fitted"].index[-1].date()}')
print(f'Residual shape: {decomp["residuals"].shape}')

## Cell 5 — Factor Returns and Rolling Sums

In [ ]:
# daily factor returns
factor_returns = decomp['scores_full'].diff()

# rolling sums keyed by window
roll_factor = {}
for w in ROLLING_WINDOWS:
    roll_factor[w] = factor_returns.rolling(w).sum()

print(f'Factor returns computed. Rolling windows: {list(roll_factor.keys())}')

## Cell 6 — Variance Explained Summary

In [ ]:
n = pca['n_pcs']
pcs = [f'PC{i+1}' for i in range(n)]

# variance table
var_df = pd.DataFrame({
    'Var Explained (%)':     (pca['var_explained'][:n] * 100).round(1),
    'Cum Var Explained (%)': (pca['cum_var_explained'][:n] * 100).round(1),
}, index=pcs)
print('=== Variance Explained ===')
print(var_df.to_string())

print('\n=== Loadings (country x PC) ===')
print(pca['loadings'].round(4).to_string())

## Cell 7 — Main Diagnostic Chart (FOCUS_COUNTRY)

In [ ]:
def plot_residual_bands(ax, dates, resid, sd_window, sd_bands, colors_inner, colors_outer):
    # rolling SD
    roll_sd = resid.rolling(sd_window, min_periods=sd_window // 2).std()
    inner, outer = sd_bands[0], sd_bands[1]
    # shade breach regions
    for i in range(len(dates) - 1):
        z = resid.iloc[i] / roll_sd.iloc[i] if roll_sd.iloc[i] > 0 else 0
        x0, x1 = dates[i], dates[i + 1]
        if abs(z) > outer:
            col = colors_outer[1] if z > 0 else colors_outer[0]
            ax.axvspan(x0, x1, color=col, alpha=0.35, linewidth=0)
        elif abs(z) > inner:
            col = colors_inner[1] if z > 0 else colors_inner[0]
            ax.axvspan(x0, x1, color=col, alpha=0.25, linewidth=0)
    # band lines
    for thresh in sd_bands:
        ax.plot(dates, roll_sd * thresh,  color='tomato',    lw=0.8, ls='--', alpha=0.7)
        ax.plot(dates, -roll_sd * thresh, color='steelblue', lw=0.8, ls='--', alpha=0.7)
    ax.axhline(0, color='black', lw=0.8)
    ax.plot(dates, resid, color='black', lw=1.2, label='Residual')
    return roll_sd

# colors: [below band (tight), above band (wide)]
COLORS_INNER = ['lightblue',  'lightsalmon']
COLORS_OUTER = ['dodgerblue', 'red']

country  = FOCUS_COUNTRY
actual   = df_spreads[country].dropna()
fitted   = decomp['fitted'][country]
resid    = decomp['residuals'][country]
contrib  = decomp['contributions'][country]
means_c  = pca['means'][country]
pcs_list = pca['loadings'].columns.tolist()

# align all to common index
idx = actual.index.intersection(fitted.index).intersection(resid.index)
actual  = actual.loc[idx]
fitted  = fitted.loc[idx]
resid   = resid.loc[idx]
contrib = contrib.loc[idx]

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle(
    f'{country} | {TENOR} Spread vs UST | PCA from {LOOKBACK_START or "full history"}',
    fontsize=13, fontweight='bold'
)

# ---- top panel: actual vs contributions stack ----
ax = axes[0]
pc_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

# stacked area: base = means_c, layers = contributions per PC
base = pd.Series(means_c, index=idx)
pos_stack = base.copy()
neg_stack = base.copy()
for k, pc in enumerate(pcs_list):
    c_vals = contrib[pc]
    pos = c_vals.clip(lower=0)
    neg = c_vals.clip(upper=0)
    ax.fill_between(idx, pos_stack, pos_stack + pos, color=pc_colors[k], alpha=0.65, label=pc)
    ax.fill_between(idx, neg_stack + neg, neg_stack, color=pc_colors[k], alpha=0.65)
    pos_stack = pos_stack + pos
    neg_stack = neg_stack + neg

# mean baseline
ax.axhline(means_c, color='gray', lw=0.8, ls=':', label=f'Mean ({means_c:.0f} bps)')
# actual spread
ax.plot(idx, actual, color='black', lw=2.0, label='Actual', zorder=5)
ax.set_ylabel('Spread (bps)')
ax.legend(loc='upper left', fontsize=8, ncol=len(pcs_list) + 2)

# ---- bottom panel: residual with bands ----
ax2 = axes[1]
plot_residual_bands(ax2, idx, resid, SD_WINDOW, SD_BANDS, COLORS_INNER, COLORS_OUTER)
ax2.set_ylabel('Residual (bps)')
ax2.set_xlabel('Date')
# legend patches for shading
patches = [
    mpatches.Patch(color='lightsalmon', alpha=0.5,  label=f'>{SD_BANDS[0]}σ wide'),
    mpatches.Patch(color='red',         alpha=0.5,  label=f'>{SD_BANDS[1]}σ wide'),
    mpatches.Patch(color='lightblue',   alpha=0.5,  label=f'>{SD_BANDS[0]}σ tight'),
    mpatches.Patch(color='dodgerblue',  alpha=0.5,  label=f'>{SD_BANDS[1]}σ tight'),
]
ax2.legend(handles=patches, loc='upper left', fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

## Cell 8 — All Countries Residual Dashboard

In [ ]:
countries_list = list(COUNTRIES.keys())
n_c = len(countries_list)
ncols = min(3, n_c)
nrows = (n_c + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), sharex=False)
axes = np.array(axes).flatten()
fig.suptitle(f'All Countries — {TENOR} Spread Residuals vs UST', fontsize=13, fontweight='bold')

for i, c in enumerate(countries_list):
    ax  = axes[i]
    res = decomp['residuals'][c].dropna()
    plot_residual_bands(ax, res.index, res, SD_WINDOW, SD_BANDS, COLORS_INNER, COLORS_OUTER)
    ax.set_title(c, fontsize=10, fontweight='bold')
    ax.set_ylabel('Residual (bps)', fontsize=8)
    ax.tick_params(axis='x', labelsize=7)

# hide any unused axes
for j in range(n_c, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Cell 9 — PC Scores in Levels with Regime Shading

In [ ]:
TRAIL_WINDOW = 60   # trailing mean window in days
scores_full  = decomp['scores_full']
n_pcs_plot   = pca['n_pcs']
pcs_list     = [f'PC{i+1}' for i in range(n_pcs_plot)]

fig, axes = plt.subplots(n_pcs_plot, 1, figsize=(14, 4 * n_pcs_plot), sharex=True)
if n_pcs_plot == 1:
    axes = [axes]
fig.suptitle(f'PC Scores — Levels and Regime ({TENOR})', fontsize=13, fontweight='bold')

for i, pc in enumerate(pcs_list):
    ax  = axes[i]
    s   = scores_full[pc].dropna()
    tr  = s.rolling(TRAIL_WINDOW, min_periods=TRAIL_WINDOW // 2).mean()
    idx = s.index
    # regime shading
    for j in range(len(idx) - 1):
        x0, x1 = idx[j], idx[j + 1]
        if pd.isna(tr.iloc[j]):
            continue
        col = '#b8e6b8' if s.iloc[j] > tr.iloc[j] else '#f5b8b8'
        ax.axvspan(x0, x1, color=col, alpha=0.4, linewidth=0)
    ax.plot(idx, s,  color='#2d5f8a', lw=1.5, label=pc)
    ax.plot(idx, tr, color='darkorange', lw=1.2, ls='--', label=f'{TRAIL_WINDOW}d mean')
    ax.axhline(0, color='black', lw=0.7)
    ax.set_ylabel('Score (bps)', fontsize=9)
    ax.set_title(pc, fontsize=10)
    ax.legend(loc='upper left', fontsize=8)
    # regime legend
    p1 = mpatches.Patch(color='#b8e6b8', alpha=0.6, label='Above trailing mean')
    p2 = mpatches.Patch(color='#f5b8b8', alpha=0.6, label='Below trailing mean')
    ax.legend(handles=[p1, p2] + ax.get_legend_handles_labels()[0][:2],
              fontsize=8, loc='upper left')

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

## Cell 10 — Factor Returns Momentum

In [ ]:
roll_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, axes = plt.subplots(n_pcs_plot, 1, figsize=(14, 4 * n_pcs_plot), sharex=True)
if n_pcs_plot == 1:
    axes = [axes]
fig.suptitle(f'Factor Return Momentum — Rolling Sums ({TENOR})', fontsize=13, fontweight='bold')

for i, pc in enumerate(pcs_list):
    ax = axes[i]
    for k, w in enumerate(ROLL_DISPLAY):
        rs = roll_factor[w][pc].dropna()
        ax.plot(rs.index, rs, color=roll_colors[k % len(roll_colors)],
                lw=1.4, label=f'{w}d sum')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('bps', fontsize=9)
    ax.set_title(pc, fontsize=10)
    ax.legend(loc='upper left', fontsize=8)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

## Cell 11 — Current Snapshot Table

In [ ]:
# build snapshot for most recent common date
last_date = decomp['residuals'].dropna(how='all').index[-1]

rows = []
for c in countries_list:
    act = decomp['fitted'].loc[last_date, c] + decomp['residuals'].loc[last_date, c]
    fit = decomp['fitted'].loc[last_date, c]
    res = decomp['residuals'].loc[last_date, c]
    # rolling SD for z-score
    r_series = decomp['residuals'][c].dropna()
    roll_sd  = r_series.rolling(SD_WINDOW, min_periods=SD_WINDOW // 2).std()
    sd_last  = roll_sd.loc[last_date] if last_date in roll_sd.index else np.nan
    z        = res / sd_last if sd_last and sd_last > 0 else np.nan
    # per-PC contributions
    pc_vals = {pc: decomp['contributions'][c].loc[last_date, pc] for pc in pcs_list}
    row = {'Country': c, 'Actual (bps)': act, 'Fitted (bps)': fit,
           'Residual (bps)': res, 'Resid Z': z}
    row.update({f'{pc} contrib': v for pc, v in pc_vals.items()})
    rows.append(row)

snap = pd.DataFrame(rows).set_index('Country')
print(f'=== Snapshot as of {last_date.date()} | Tenor: {TENOR} ===')
print(snap.round(1).to_string())